In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier,
    BaggingClassifier,
)
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from itertools import combinations, product
import warnings

warnings.filterwarnings("ignore")

In [2]:
# Constants 
RandomSTATE = 1082
TEST_SIZE = 0.2
max_base_models = 3

In [3]:
# Load the dataset
try:
    df = pd.read_csv("../../Datasets/csv/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../Datasets/csv/spoofing-merged-gps-only.csv' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [4]:
# Data Preprocessing
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
drop_cols = ['timestamp', 'label']
X = df.drop(columns=drop_cols)
y = df['label']

if y.dtype == object or y.dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RandomSTATE, stratify=y
)


In [5]:
# Define all possible base and meta models
base_model_options = [
    ('lr', LogisticRegression(random_state=RandomSTATE, max_iter=1200, solver='saga')),
    ('ridge', RidgeClassifier(random_state=RandomSTATE)),
    ('rf', RandomForestClassifier(random_state=RandomSTATE)),
    ('gb', GradientBoostingClassifier(random_state=RandomSTATE)),
    ('ada', AdaBoostClassifier(random_state=RandomSTATE)),
    ('et', ExtraTreesClassifier(random_state=RandomSTATE)),
    ('hgb', HistGradientBoostingClassifier(random_state=RandomSTATE)),
    ('svc', SVC(probability=True, random_state=RandomSTATE)),
    ('lsvc', LinearSVC(random_state=RandomSTATE, max_iter=2000)),
    ('knn', KNeighborsClassifier()),
    ('gnb', GaussianNB()),
    ('bnb', BernoulliNB()),
    ('dt', DecisionTreeClassifier(random_state=RandomSTATE)),
    ('bagging', BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RandomSTATE), random_state=RandomSTATE)),
]

meta_model_options = [
    LogisticRegression(random_state=RandomSTATE, max_iter=1200, solver='saga'),
    RidgeClassifier(random_state=RandomSTATE),
    RandomForestClassifier(random_state=RandomSTATE),
    GradientBoostingClassifier(random_state=RandomSTATE),
    AdaBoostClassifier(random_state=RandomSTATE),
    ExtraTreesClassifier(random_state=RandomSTATE),
    HistGradientBoostingClassifier(random_state=RandomSTATE),
    SVC(probability=True, random_state=RandomSTATE),
    KNeighborsClassifier(),
    GaussianNB(),
    BernoulliNB(),
    DecisionTreeClassifier(random_state=RandomSTATE),
    BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RandomSTATE), random_state=RandomSTATE),
]


In [6]:
def run_stacking_experiments(X_train, X_test, y_train, y_test, base_model_options, meta_model_options, max_base_models=3):
    results = []
    total_runs = 0
    for n in range(2, max_base_models + 1):
        for base_model_combo in combinations(base_model_options, n):
            base_names = [name for name, _ in base_model_combo]
            if len(set(base_names)) != len(base_names):
                continue
            for meta_model in meta_model_options:
                try:
                    stacking_model = StackingClassifier(
                        estimators=list(base_model_combo),
                        final_estimator=meta_model,
                        cv=5,
                        passthrough=False,
                        n_jobs=-1
                    )
                    stacking_model.fit(X_train, y_train)
                    y_pred = stacking_model.predict(X_test)
                    acc = accuracy_score(y_test, y_pred)
                    results.append({
                        "base_models": base_names,
                        "meta_model": meta_model.__class__.__name__,
                        "accuracy": acc,
                        "confusion_matrix": confusion_matrix(y_test, y_pred),
                        "classification_report": classification_report(y_test, y_pred, zero_division=0, output_dict=False)
                    })
                    total_runs += 1
                    print(f"[{total_runs}] Stack: {base_names} | Meta: {meta_model.__class__.__name__} | Accuracy: {acc:.4f}")
                except Exception as e:
                    print(f"Skipped {base_names} -> {meta_model.__class__.__name__} due to error: {e}")
    return results

In [7]:
if __name__ == "__main__":
    results = run_stacking_experiments(
        X_train, X_test, y_train, y_test,
        base_model_options, meta_model_options,
        max_base_models=max_base_models
    )
    if results:
        best = max(results, key=lambda x: x["accuracy"])
        print("\nBest Stacking Model:")
        print(f"Base models: {best['base_models']}")
        print(f"Meta model: {best['meta_model']}")
        print(f"Test Accuracy: {best['accuracy']:.4f}")
        print("Confusion Matrix:\n", best["confusion_matrix"])
        print("Classification Report:\n", best["classification_report"])
    else:
        print("No stacking models succeeded.")

[1] Stack: ['lr', 'ridge'] | Meta: LogisticRegression | Accuracy: 1.0000
[2] Stack: ['lr', 'ridge'] | Meta: RidgeClassifier | Accuracy: 1.0000
[3] Stack: ['lr', 'ridge'] | Meta: RandomForestClassifier | Accuracy: 1.0000
[4] Stack: ['lr', 'ridge'] | Meta: GradientBoostingClassifier | Accuracy: 1.0000
[5] Stack: ['lr', 'ridge'] | Meta: AdaBoostClassifier | Accuracy: 1.0000
[6] Stack: ['lr', 'ridge'] | Meta: ExtraTreesClassifier | Accuracy: 1.0000
[7] Stack: ['lr', 'ridge'] | Meta: HistGradientBoostingClassifier | Accuracy: 1.0000
[8] Stack: ['lr', 'ridge'] | Meta: SVC | Accuracy: 1.0000
[9] Stack: ['lr', 'ridge'] | Meta: KNeighborsClassifier | Accuracy: 1.0000
[10] Stack: ['lr', 'ridge'] | Meta: GaussianNB | Accuracy: 1.0000
[11] Stack: ['lr', 'ridge'] | Meta: BernoulliNB | Accuracy: 1.0000
[12] Stack: ['lr', 'ridge'] | Meta: DecisionTreeClassifier | Accuracy: 1.0000
[13] Stack: ['lr', 'rf'] | Meta: LogisticRegression | Accuracy: 1.0000
[14] Stack: ['lr', 'rf'] | Meta: RidgeClassifier | 

KeyboardInterrupt: 